In [62]:
# Cell 1
import pandas as pd
import numpy as np
import joblib
import pickle
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
from lightgbm import LGBMClassifier

In [63]:
# Cell 2
original_df = pd.read_csv('../data/processed_data.csv')
new_df = pd.read_csv('../data/final_bn_data.csv')

if 'content' in new_df.columns and 'text' not in new_df.columns:
    new_df = new_df.rename(columns={'content': 'text'})

combined_df = pd.concat([
    original_df[['text', 'label']],
    new_df[['text', 'label']]
], ignore_index=True)

combined_df['label'] = combined_df['label'].astype(int)

In [64]:
# Cell 3
def preprocess_bangla_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[a-zA-Z0-9]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[#@]\w+', '', text)
    text = re.sub(r'[\r\n\t]', ' ', text)
    return text


combined_df['cleaned_text'] = combined_df['text'].apply(preprocess_bangla_text)

In [65]:
# Cell 4
tfidf_vectorizer = TfidfVectorizer(
    max_features=6000,
    lowercase=False,
    min_df=3,
    max_df=0.8,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = tfidf_vectorizer.fit_transform(combined_df['cleaned_text'])
y = combined_df['label']

joblib.dump(tfidf_vectorizer, 'final_vectorizer.pkl')

['final_vectorizer.pkl']

In [66]:
# Cell 5
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [67]:
# Cell 6
nb_model = MultinomialNB(alpha=0.01)
nb_model.fit(X_train, y_train)

,alpha,0.01
,force_alpha,True
,fit_prior,True
,class_prior,None


In [68]:
# Cell 7
cnb_model = ComplementNB(alpha=0.01)
cnb_model.fit(X_train, y_train)

,alpha,0.01
,force_alpha,True
,fit_prior,True
,class_prior,None
,norm,False


In [69]:
# Cell 8
lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=10,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.01,
    reg_lambda=0.01,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    is_unbalance=True
)
lgb_model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,10
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [70]:
# Cell 9
pa_model = PassiveAggressiveClassifier(
    C=0.1,
    max_iter=1500,
    random_state=42,
    verbose=0
)
pa_model.fit(X_train, y_train)

,C,0.1
,fit_intercept,True
,max_iter,1500
,tol,0.001
,early_stopping,False
,validation_fraction,0.1
,n_iter_no_change,5
,shuffle,True
,verbose,0
,loss,'hinge'
,n_jobs,None


In [71]:
# Cell 10
voting_model = VotingClassifier(
    estimators=[
        ('nb', nb_model),
        ('cnb', cnb_model),
        ('lgb', lgb_model),
        ('pa', pa_model)
    ],
    voting='hard'
)
voting_model.fit(X_train, y_train)

,estimators,"[('nb', ...), ('cnb', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,alpha,0.01
,force_alpha,True
,fit_prior,True
,class_prior,None
,alpha,0.01


In [72]:
# Cell 11
joblib.dump(nb_model, 'final_naive_bayes.pkl')
joblib.dump(cnb_model, 'final_complement_nb.pkl')
joblib.dump(lgb_model, 'final_lightgbm.pkl')
joblib.dump(pa_model, 'final_passive_aggressive.pkl')
joblib.dump(voting_model, 'final_voting_ensemble.pkl')

print("All models saved successfully")

All models saved successfully


In [73]:
# Cell 12
def predict_with_all_models(text):
    cleaned = preprocess_bangla_text(text)
    features = tfidf_vectorizer.transform([cleaned])

    results = {}
    results['naive_bayes'] = nb_model.predict(features)[0]
    results['complement_nb'] = cnb_model.predict(features)[0]
    results['lightgbm'] = lgb_model.predict(features)[0]
    results['passive_aggressive'] = pa_model.predict(features)[0]
    results['voting_ensemble'] = voting_model.predict(features)[0]

    return results


with open('prediction_function.py', 'w') as f:
    f.write('''
import joblib
import re

def preprocess_bangla_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\\S+|www\\S+|https\\S+', '', text)
    text = re.sub(r'[a-zA-Z0-9]', '', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    text = re.sub(r'[#@]\\\\w+', '', text)
    text = re.sub(r'[\\\\r\\\\n\\\\t]', ' ', text)
    return text

def load_models():
    vectorizer = joblib.load('final_vectorizer.pkl')
    nb_model = joblib.load('final_naive_bayes.pkl')
    cnb_model = joblib.load('final_complement_nb.pkl')
    lgb_model = joblib.load('final_lightgbm.pkl')
    pa_model = joblib.load('final_passive_aggressive.pkl')
    voting_model = joblib.load('final_voting_ensemble.pkl')
    return vectorizer, nb_model, cnb_model, lgb_model, pa_model, voting_model

def predict_all(text):
    vectorizer, nb_model, cnb_model, lgb_model, pa_model, voting_model = load_models()
    cleaned = preprocess_bangla_text(text)
    features = vectorizer.transform([cleaned])

    results = {
        'naive_bayes': int(nb_model.predict(features)[0]),
        'complement_nb': int(cnb_model.predict(features)[0]),
        'lightgbm': int(lgb_model.predict(features)[0]),
        'passive_aggressive': int(pa_model.predict(features)[0]),
        'voting_ensemble': int(voting_model.predict(features)[0])
    }
    return results
''')

print("Prediction function saved to prediction_function.py")

Prediction function saved to prediction_function.py


In [74]:
# Cell 13
sample_real = combined_df[combined_df['label'] == 1].iloc[0]['text'][:1000]
sample_fake = combined_df[combined_df['label'] == 0].iloc[0]['text'][:1000]

test_text_real = sample_real + "..."
test_text_fake = sample_fake + "..."

print("Testing with REAL news sample:", test_text_real)
real_results = predict_with_all_models(test_text_real)
print("Real news prediction:", real_results)

print("\nTesting with FAKE news sample:", test_text_fake)
fake_results = predict_with_all_models(test_text_fake)
print("Fake news prediction:", fake_results)

Testing with REAL news sample: হঠাৎ আফগান ক্রিকেট বোর্ড প্রধানের পদত্যাগ ক্রিকেট বিশ্বের নতুন চমকের নাম আফগানিস্তান। কয়েক বছরে তাদের পারফরম্যান্স নজর কেড়েছে ক্রিকেট জগতের। এশিয়া কাপের ১৪তম আসরেও দারুণ ছন্দে আছে আফগান ক্রিকেটাররা। পাঁচবারের এশিয়া কাপ চ্যাম্পিয়ন শ্রীলঙ্কাকে হারিয়ে এরই মধ্যে সেরা চারে পৌঁছে গেছে তারা। তবে দলের এমন ভালো সময়ে হঠাৎ করেই পদত্যাগ করেন আফগান ক্রিকেট বোর্ড (এসিবি) প্রধান আতিফ মাশাল। তবে হঠাৎ কী কারণে তার এই পদত্যাগ- এ নিয়ে ওঠা প্রশ্নের জবাব দিয়েছেন আতিফ। তিনি জানান, সরকারের অন্য একটি বিশেষ পদে তাকে নিয়োগ দেওয়ার কারণেই বোর্ডের দায়িত্ব থেকে অব্যাহতি। তার পরিবর্তে এসিবির দায়িত্ব নেবেন আফগান বোর্ডের সাবেক সহ সভাপতি আজিজ উল্লাহ ফজলে।   ২০১৭ সালের জানুয়ারিতে পাঁচ বছরের জন্য আতিফকে বোর্ড প্রধান নির্বাচন করা হলেও দায়িত্বের আড়াই বছরের মাথায় দায়িত্ব থেকে অব্যাহতি দিলেন তিনি। বিডি প্রতিদিন/ ওয়াসিফ...
Real news prediction: {'naive_bayes': np.int64(1), 'complement_nb': np.int64(0), 'lightgbm': np.int64(1), 'passive_aggressive': np.int64(1), 'voting_ensemble': np.int64(1)}



In [75]:
# Cell 14
new_df_test = pd.read_csv('../data/final_bn_data.csv')
if 'content' in new_df_test.columns:
    new_df_test = new_df_test.rename(columns={'content': 'text'})

new_df_test['cleaned_text'] = new_df_test['text'].apply(preprocess_bangla_text)
X_new = tfidf_vectorizer.transform(new_df_test['cleaned_text'])
y_new_true = new_df_test['label'].astype(int)

lgb_predictions = lgb_model.predict(X_new)
lgb_accuracy = accuracy_score(y_new_true, lgb_predictions)
print(f"LightGBM accuracy on new data: {lgb_accuracy:.4f}")

LightGBM accuracy on new data: 0.8857


In [76]:
# Cell 15

models_dict = {
    'Naive Bayes': nb_model,
    'Complement NB': cnb_model,
    'LightGBM': lgb_model,
    'Passive Aggressive': pa_model,
    'Voting Ensemble': voting_model
}

test_results = {}
for name, model in models_dict.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    test_results[name] = acc
    print(f"{name:<20}: {acc:.4f}")

print("\nModel performance on new dataset:")

new_df_test = pd.read_csv('../data/final_bn_data.csv')
if 'content' in new_df_test.columns:
    new_df_test = new_df_test.rename(columns={'content': 'text'})

new_df_test['cleaned_text'] = new_df_test['text'].apply(preprocess_bangla_text)
X_new = tfidf_vectorizer.transform(new_df_test['cleaned_text'])
y_new_true = new_df_test['label'].astype(int)

new_results = {}
for name, model in models_dict.items():
    y_new_pred = model.predict(X_new)
    acc = accuracy_score(y_new_true, y_new_pred)
    new_results[name] = acc
    print(f"{name:<20}: {acc:.4f}")

Naive Bayes         : 0.7882
Complement NB       : 0.7064
LightGBM            : 0.7051
Passive Aggressive  : 0.7895
Voting Ensemble     : 0.7385

Model performance on new dataset:
Naive Bayes         : 0.7283
Complement NB       : 0.6838
LightGBM            : 0.8857
Passive Aggressive  : 0.7914
Voting Ensemble     : 0.7737


In [77]:
print("Improving other models...")

print("\n1. Improving Naive Bayes...")
best_nb_acc = 0
best_nb_alpha = 0.01
for alpha in [0.001, 0.01, 0.1, 0.5, 1.0]:
    nb_temp = MultinomialNB(alpha=alpha)
    nb_temp.fit(X_train, y_train)
    y_new_pred = nb_temp.predict(X_new)
    acc = accuracy_score(y_new_true, y_new_pred)
    if acc > best_nb_acc:
        best_nb_acc = acc
        best_nb_alpha = alpha

nb_improved = MultinomialNB(alpha=best_nb_alpha)
nb_improved.fit(X_train, y_train)
print(f"Best alpha: {best_nb_alpha}, Accuracy: {best_nb_acc:.4f}")

print("\n2. Improving Complement NB...")
best_cnb_acc = 0
best_cnb_alpha = 0.01
for alpha in [0.001, 0.01, 0.1, 0.5, 1.0, 2.0]:
    cnb_temp = ComplementNB(alpha=alpha)
    cnb_temp.fit(X_train, y_train)
    y_new_pred = cnb_temp.predict(X_new)
    acc = accuracy_score(y_new_true, y_new_pred)
    if acc > best_cnb_acc:
        best_cnb_acc = acc
        best_cnb_alpha = alpha

cnb_improved = ComplementNB(alpha=best_cnb_alpha)
cnb_improved.fit(X_train, y_train)
print(f"Best alpha: {best_cnb_alpha}, Accuracy: {best_cnb_acc:.4f}")

print("\n3. Improving Passive Aggressive...")
best_pa_acc = 0
best_pa_C = 0.1
for C in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0]:
    pa_temp = PassiveAggressiveClassifier(C=C, max_iter=1500, random_state=42, verbose=0)
    pa_temp.fit(X_train, y_train)
    y_new_pred = pa_temp.predict(X_new)
    acc = accuracy_score(y_new_true, y_new_pred)
    if acc > best_pa_acc:
        best_pa_acc = acc
        best_pa_C = C

pa_improved = PassiveAggressiveClassifier(C=best_pa_C, max_iter=1500, random_state=42, verbose=0)
pa_improved.fit(X_train, y_train)
print(f"Best C: {best_pa_C}, Accuracy: {best_pa_acc:.4f}")

print("\n4. Updating Voting Ensemble...")
voting_improved = VotingClassifier(
    estimators=[
        ('nb', nb_improved),
        ('cnb', cnb_improved),
        ('lgb', lgb_model),
        ('pa', pa_improved)
    ],
    voting='hard'
)
voting_improved.fit(X_train, y_train)

print("\n5. Saving improved models...")
joblib.dump(nb_improved, 'improved_naive_bayes.pkl')
joblib.dump(cnb_improved, 'improved_complement_nb.pkl')
joblib.dump(pa_improved, 'improved_passive_aggressive.pkl')
joblib.dump(voting_improved, 'improved_voting_ensemble.pkl')

print("Improved models saved successfully")

Improving other models...

1. Improving Naive Bayes...
Best alpha: 1.0, Accuracy: 0.7290

2. Improving Complement NB...
Best alpha: 2.0, Accuracy: 0.6885

3. Improving Passive Aggressive...
Best C: 0.5, Accuracy: 0.8300

4. Updating Voting Ensemble...

5. Saving improved models...
Improved models saved successfully


In [78]:
print("Testing improved models on new dataset:")

improved_models = {
    'Naive Bayes Improved': nb_improved,
    'Complement NB Improved': cnb_improved,
    'Passive Aggressive Improved': pa_improved,
    'Voting Ensemble Improved': voting_improved,
    'LightGBM (original)': lgb_model
}

improved_results = {}
for name, model in improved_models.items():
    y_new_pred = model.predict(X_new)
    acc = accuracy_score(y_new_true, y_new_pred)
    improved_results[name] = acc
    print(f"{name:<30}: {acc:.4f}")

print("\nComparison with original models:")
print(f"{'Original Naive Bayes':<30}: {new_results['Naive Bayes']:.4f}")
print(f"{'Improved Naive Bayes':<30}: {improved_results['Naive Bayes Improved']:.4f}")
print(f"{'Difference':<30}: {improved_results['Naive Bayes Improved'] - new_results['Naive Bayes']:+.4f}")
print()
print(f"{'Original Complement NB':<30}: {new_results['Complement NB']:.4f}")
print(f"{'Improved Complement NB':<30}: {improved_results['Complement NB Improved']:.4f}")
print(f"{'Difference':<30}: {improved_results['Complement NB Improved'] - new_results['Complement NB']:+.4f}")
print()
print(f"{'Original Passive Aggressive':<30}: {new_results['Passive Aggressive']:.4f}")
print(f"{'Improved Passive Aggressive':<30}: {improved_results['Passive Aggressive Improved']:.4f}")
print(f"{'Difference':<30}: {improved_results['Passive Aggressive Improved'] - new_results['Passive Aggressive']:+.4f}")
print()
print(f"{'Original Voting Ensemble':<30}: {new_results['Voting Ensemble']:.4f}")
print(f"{'Improved Voting Ensemble':<30}: {improved_results['Voting Ensemble Improved']:.4f}")
print(f"{'Difference':<30}: {improved_results['Voting Ensemble Improved'] - new_results['Voting Ensemble']:+.4f}")

Testing improved models on new dataset:
Naive Bayes Improved          : 0.7290
Complement NB Improved        : 0.6885
Passive Aggressive Improved   : 0.8300
Voting Ensemble Improved      : 0.8010
LightGBM (original)           : 0.8857

Comparison with original models:
Original Naive Bayes          : 0.7283
Improved Naive Bayes          : 0.7290
Difference                    : +0.0006

Original Complement NB        : 0.6838
Improved Complement NB        : 0.6885
Difference                    : +0.0047

Original Passive Aggressive   : 0.7914
Improved Passive Aggressive   : 0.8300
Difference                    : +0.0385

Original Voting Ensemble      : 0.7737
Improved Voting Ensemble      : 0.8010
Difference                    : +0.0272


In [80]:
# Weighted Voting based on model accuracy

print("Creating accuracy-weighted voting ensemble...")

accuracies = {}
predictions = {}

models_for_weighting = {
    'nb': nb_improved,
    'cnb': cnb_improved,
    'lgb': lgb_model,
    'pa': pa_improved
}

for name, model in models_for_weighting.items():
    preds = model.predict(X_new)
    predictions[name] = preds
    acc = accuracy_score(y_new_true, preds)
    accuracies[name] = acc
    print(f"{name} accuracy: {acc:.4f}")

total_accuracy = sum(accuracies.values())
weights = {name: acc/total_accuracy for name, acc in accuracies.items()}

print("\nModel weights based on accuracy:")
for name, weight in weights.items():
    print(f"{name}: {weight:.3f} (accuracy: {accuracies[name]:.4f})")

weighted_sum = np.zeros(len(y_new_true))
for name, preds in predictions.items():
    weighted_sum += weights[name] * preds

weighted_predictions = (weighted_sum > 0.5).astype(int)
weighted_accuracy = accuracy_score(y_new_true, weighted_predictions)

print(f"\nAccuracy-weighted voting accuracy: {weighted_accuracy:.4f}")

from sklearn.base import BaseEstimator, ClassifierMixin

class AccuracyWeightedVoting(BaseEstimator, ClassifierMixin):
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights
        self.model_names = list(models.keys())

    def fit(self, X, y):
        for name, model in self.models.items():
            model.fit(X, y)


        if self.weights is None:
            from sklearn.model_selection import cross_val_score
            self.weights = {}
            for name, model in self.models.items():
                scores = cross_val_score(model, X, y, cv=3, scoring='accuracy')
                self.weights[name] = np.mean(scores)

            # Normalize weights
            total = sum(self.weights.values())
            self.weights = {k: v/total for k, v in self.weights.items()}

        return self

    def predict(self, X):
        all_preds = []
        for name, model in self.models.items():
            preds = model.predict(X)
            all_preds.append(preds * self.weights[name])

        weighted_sum = np.sum(all_preds, axis=0)

        return (weighted_sum > 0.5).astype(int)

    def predict_proba(self, X):

        all_probas = []
        for name, model in self.models.items():
            if hasattr(model, 'predict_proba'):
                probas = model.predict_proba(X)[:, 1]
            else:
                preds = model.predict(X)
                probas = preds.astype(float)

            all_probas.append(probas * self.weights[name])

        # Weighted average
        weighted_proba = np.sum(all_probas, axis=0)

        # Return as 2D array for consistency
        return np.column_stack([1-weighted_proba, weighted_proba])


print("\nCreating accuracy-weighted voting classifier...")

precalc_weights = weights

weighted_voter = AccuracyWeightedVoting(
    models=models_for_weighting,
    weights=precalc_weights
)


weighted_voter.models = models_for_weighting
weighted_voter.weights = precalc_weights

# Test
weighted_voter_preds = weighted_voter.predict(X_new)
weighted_voter_acc = accuracy_score(y_new_true, weighted_voter_preds)
print(f"AccuracyWeightedVoting accuracy: {weighted_voter_acc:.4f}")

print("\nComparison of all ensemble methods:")
print(f"Hard Voting (equal weights)         : {improved_results['Voting Ensemble Improved']:.4f}")
print(f"Accuracy-Weighted Voting           : {weighted_voter_acc:.4f}")

joblib.dump(weighted_voter, 'accuracy_weighted_voting.pkl')
print("\nSaved accuracy_weighted_voting.pkl")


weights_info = {
    'model_names': list(models_for_weighting.keys()),
    'weights': precalc_weights,
    'accuracies': accuracies
}
joblib.dump(weights_info, 'voting_weights_info.pkl')
print("Saved voting_weights_info.pkl")

Creating accuracy-weighted voting ensemble...
nb accuracy: 0.7290
cnb accuracy: 0.6885
lgb accuracy: 0.8857
pa accuracy: 0.8300

Model weights based on accuracy:
nb: 0.233 (accuracy: 0.7290)
cnb: 0.220 (accuracy: 0.6885)
lgb: 0.283 (accuracy: 0.8857)
pa: 0.265 (accuracy: 0.8300)

Accuracy-weighted voting accuracy: 0.8364

Creating accuracy-weighted voting classifier...
AccuracyWeightedVoting accuracy: 0.8364

Comparison of all ensemble methods:
Hard Voting (equal weights)         : 0.8010
Accuracy-Weighted Voting           : 0.8364

Saved accuracy_weighted_voting.pkl
Saved voting_weights_info.pkl


In [81]:
#Final comparison of all models

print("FINAL MODEL COMPARISON ON NEW DATASET")
print("="*60)

final_results = {
    'Naive Bayes': improved_results['Naive Bayes Improved'],
    'Complement NB': improved_results['Complement NB Improved'],
    'LightGBM': improved_results['LightGBM (original)'],
    'Passive Aggressive': improved_results['Passive Aggressive Improved'],
    'Hard Voting Ensemble': improved_results['Voting Ensemble Improved'],
    'Accuracy-Weighted Voting': weighted_voter_acc
}

# Sort by accuracy
sorted_results = sorted(final_results.items(), key=lambda x: x[1], reverse=True)

for i, (model_name, accuracy) in enumerate(sorted_results):
    rank = i + 1
    print(f"{rank}. {model_name:<25}: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("="*60)

print("\nSaving all final models for frontend integration...")

final_models = {
    'naive_bayes': nb_improved,
    'complement_nb': cnb_improved,
    'lightgbm': lgb_model,
    'passive_aggressive': pa_improved,
    'hard_voting': voting_improved,
    'weighted_voting': weighted_voter
}

for name, model in final_models.items():
    filename = f"frontend_{name}.pkl"
    joblib.dump(model, filename)
    print(f"Saved: {filename}")


FINAL MODEL COMPARISON ON NEW DATASET
1. LightGBM                 : 0.8857 (88.57%)
2. Accuracy-Weighted Voting : 0.8364 (83.64%)
3. Passive Aggressive       : 0.8300 (83.00%)
4. Hard Voting Ensemble     : 0.8010 (80.10%)
5. Naive Bayes              : 0.7290 (72.90%)
6. Complement NB            : 0.6885 (68.85%)

Saving all final models for frontend integration...
Saved: frontend_naive_bayes.pkl
Saved: frontend_complement_nb.pkl
Saved: frontend_lightgbm.pkl
Saved: frontend_passive_aggressive.pkl
Saved: frontend_hard_voting.pkl
Saved: frontend_weighted_voting.pkl
